# CS7 S03: CPU-GPU Workload Partitioning

In [4]:
!nvidia-smi
!nvcc --version
!nproc
!free -g
!pip install -q python-docx

Mon Oct 19 16:07:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile s03_bench.cu
// S03 - CPU-GPU workload partitioning benchmark (CUDA + OpenMP)
//
// Workload: N independent image tiles, each T x T floats. Every tile is processed by P passes of a
// 3x3 box blur (clamped edges inside the tile). Tiles are independent, so any subset can run on
// either device. A fraction f_cpu of the tiles goes to the CPU (OpenMP), the rest to the GPU (CUDA).
//   f_cpu = 1.0 -> CPU-only baseline,  f_cpu = 0.0 -> GPU-only baseline,  otherwise hybrid.
//
// The GPU work (H2D copy, P kernels, D2H copy, in chunks) is enqueued asynchronously first, then the
// host runs its own share with OpenMP while the GPU works, and finally waits for the GPU.
// Raw per-run measurements are printed as CSV rows on stdout, one row per (cpu_frac, rep).
//
// Build (GPU):   nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native s03_bench.cu -o s03_bench
// Build (no GPU, pipeline test only):  g++ -O3 -fopenmp -march=native -DNO_CUDA -x c++ s03_bench.cu -o s03_bench_cpu
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <cstdint>
#include <cmath>
#include <vector>
#include <string>
#include <algorithm>
#include <omp.h>

#ifndef NO_CUDA
#include <cuda_runtime.h>
#else
// ---- host-only shim: lets the whole pipeline be tested on a machine without a GPU.
// The "GPU" runs synchronously on the host, so timings from this build say nothing about real hardware.
typedef int cudaError_t;
static const cudaError_t cudaSuccess = 0;
typedef int cudaStream_t;
struct ShimEvent { double t; };
typedef ShimEvent* cudaEvent_t;
enum cudaMemcpyKind { cudaMemcpyHostToDevice, cudaMemcpyDeviceToHost };
#define cudaHostAllocDefault 0
static const char* cudaGetErrorString(cudaError_t) { return "no error"; }
static cudaError_t cudaMalloc(void** p, size_t n) { *p = malloc(n); return *p ? 0 : 2; }
static cudaError_t cudaFree(void* p) { free(p); return 0; }
static cudaError_t cudaHostAlloc(void** p, size_t n, unsigned) { *p = malloc(n); return *p ? 0 : 2; }
static cudaError_t cudaFreeHost(void* p) { free(p); return 0; }
static cudaError_t cudaStreamCreate(cudaStream_t* s) { *s = 0; return 0; }
static cudaError_t cudaStreamDestroy(cudaStream_t) { return 0; }
static cudaError_t cudaEventCreate(cudaEvent_t* e) { *e = new ShimEvent{0.0}; return 0; }
static cudaError_t cudaEventDestroy(cudaEvent_t e) { delete e; return 0; }
static cudaError_t cudaEventRecord(cudaEvent_t e, cudaStream_t) { e->t = omp_get_wtime() * 1e3; return 0; }
static cudaError_t cudaEventElapsedTime(float* ms, cudaEvent_t a, cudaEvent_t b) { *ms = (float)(b->t - a->t); return 0; }
static cudaError_t cudaStreamWaitEvent(cudaStream_t, cudaEvent_t, unsigned) { return 0; }
static cudaError_t cudaStreamSynchronize(cudaStream_t) { return 0; }
static cudaError_t cudaDeviceSynchronize() { return 0; }
static cudaError_t cudaGetLastError() { return 0; }
static cudaError_t cudaMemcpyAsync(void* d, const void* s, size_t n, cudaMemcpyKind, cudaStream_t) { memcpy(d, s, n); return 0; }
#endif

#define CK(x) do { cudaError_t e_ = (x); if (e_ != cudaSuccess) { \
    fprintf(stderr, "CUDA error '%s' at %s:%d\n", cudaGetErrorString(e_), __FILE__, __LINE__); exit(1); } } while (0)

// ------------------------------------------------------------------ GPU kernel
#ifndef NO_CUDA
// 3x3 box blur on a batch of tiles; blockIdx.z selects the tile. Summation order matches the CPU code.
__global__ void k_blur(const float* __restrict__ in, float* __restrict__ out, int T) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= T || y >= T) return;
    size_t base = (size_t)blockIdx.z * T * T;
    const float* src = in + base;
    float s = 0.f;
    for (int dy = -1; dy <= 1; dy++)
        for (int dx = -1; dx <= 1; dx++) {
            int xx = min(max(x + dx, 0), T - 1), yy = min(max(y + dy, 0), T - 1);
            s += src[(size_t)yy * T + xx];
        }
    out[base + (size_t)y * T + x] = s / 9.f;
}
#endif

// ------------------------------------------------------------------ CPU kernel (same arithmetic order)
static inline void blur_edge_px(const float* r0, const float* r1, const float* r2, float* o, int T, int x) {
    int xm = x > 0 ? x - 1 : 0, xp = x < T - 1 ? x + 1 : T - 1;
    float s = 0.f;
    s += r0[xm]; s += r0[x]; s += r0[xp];
    s += r1[xm]; s += r1[x]; s += r1[xp];
    s += r2[xm]; s += r2[x]; s += r2[xp];
    o[x] = s / 9.f;
}

static void blur_tile_cpu(const float* __restrict__ in, float* __restrict__ out, int T) {
    for (int y = 0; y < T; y++) {
        int ym = y > 0 ? y - 1 : 0, yp = y < T - 1 ? y + 1 : T - 1;
        const float* r0 = in + (size_t)ym * T;
        const float* r1 = in + (size_t)y * T;
        const float* r2 = in + (size_t)yp * T;
        float* o = out + (size_t)y * T;
        blur_edge_px(r0, r1, r2, o, T, 0);
#pragma omp simd
        for (int x = 1; x < T - 1; x++) {
            float s = 0.f;
            s += r0[x - 1]; s += r0[x]; s += r0[x + 1];
            s += r1[x - 1]; s += r1[x]; s += r1[x + 1];
            s += r2[x - 1]; s += r2[x]; s += r2[x + 1];
            o[x] = s / 9.f;
        }
        blur_edge_px(r0, r1, r2, o, T, T - 1);
    }
}

// P passes, ping-pong between `out` and `tmp` so that the final pass lands in `out`.
static void cpu_tile(const float* in, float* out, float* tmp, int T, int P) {
    const float* src = in;
    for (int i = 0; i < P; i++) {
        float* dst = (((P - 1 - i) & 1) == 0) ? out : tmp;
        blur_tile_cpu(src, dst, T);
        src = dst;
    }
}

// Runs tiles [first, first+count) on the CPU with `th` OpenMP threads; per-thread busy time -> busy[id*8].
static int cpu_process(const float* h_in, float* h_out, int first, int count, int T, int P, int th, double* busy) {
    if (count <= 0) return 0;
    size_t tp = (size_t)T * T;
    int team = 0;
#pragma omp parallel num_threads(th)
    {
        static thread_local std::vector<float> tmp;
        if (tmp.size() < tp) tmp.resize(tp);
        int id = omp_get_thread_num();
        double b = 0.0;
#pragma omp single
        team = omp_get_num_threads();
#pragma omp for schedule(dynamic, 1)
        for (int k = 0; k < count; k++) {
            double t0 = omp_get_wtime();
            size_t off = (size_t)(first + k) * tp;
            cpu_tile(h_in + off, h_out + off, tmp.data(), T, P);
            b += omp_get_wtime() - t0;
        }
        busy[id * 8] = b;
    }
    return team;
}

// ------------------------------------------------------------------ GPU pipeline
struct GpuCtx {
    int S = 1, chunk = 64, maxChunks = 0;
    std::vector<cudaStream_t> st;
    std::vector<float*> dA, dB;
    std::vector<cudaEvent_t> ev;      // 4 events per chunk: before H2D, after H2D, after kernels, after D2H
    cudaEvent_t evStart = nullptr;
    std::vector<cudaEvent_t> evEnd;   // one per stream
};

static void launch_blur(const float* in, float* out, int T, int nt, cudaStream_t st) {
#ifndef NO_CUDA
    dim3 b(16, 16), g((T + 15) / 16, (T + 15) / 16, nt);
    k_blur<<<g, b, 0, st>>>(in, out, T);
#else
    (void)st;
    for (int t = 0; t < nt; t++) blur_tile_cpu(in + (size_t)t * T * T, out + (size_t)t * T * T, T);
#endif
}

static void gpu_setup(GpuCtx& g, int S, int chunk, int maxTiles, size_t tp) {
    g.S = S; g.chunk = chunk;
    g.maxChunks = (maxTiles + chunk - 1) / chunk;
    g.st.resize(S); g.dA.resize(S); g.dB.resize(S); g.evEnd.resize(S);
    size_t bytes = (size_t)chunk * tp * sizeof(float);
    for (int s = 0; s < S; s++) {
        CK(cudaStreamCreate(&g.st[s]));
        CK(cudaMalloc((void**)&g.dA[s], bytes));
        CK(cudaMalloc((void**)&g.dB[s], bytes));
        CK(cudaEventCreate(&g.evEnd[s]));
    }
    g.ev.resize((size_t)4 * g.maxChunks);
    for (auto& e : g.ev) CK(cudaEventCreate(&e));
    CK(cudaEventCreate(&g.evStart));
}

static void gpu_teardown(GpuCtx& g) {
    for (int s = 0; s < g.S; s++) {
        CK(cudaFree(g.dA[s])); CK(cudaFree(g.dB[s]));
        CK(cudaStreamDestroy(g.st[s])); CK(cudaEventDestroy(g.evEnd[s]));
    }
    for (auto& e : g.ev) CK(cudaEventDestroy(e));
    CK(cudaEventDestroy(g.evStart));
}

// Enqueue tiles [first, first+count) onto the GPU streams (returns immediately; work runs asynchronously).
static int gpu_enqueue(GpuCtx& g, const float* h_in, float* h_out, int first, int count, int T, int P) {
    size_t tp = (size_t)T * T;
    int nch = (count + g.chunk - 1) / g.chunk;
    CK(cudaEventRecord(g.evStart, g.st[0]));
    for (int s = 1; s < g.S; s++) CK(cudaStreamWaitEvent(g.st[s], g.evStart, 0));
    for (int c = 0; c < nch; c++) {
        int s = c % g.S;
        int t0 = first + c * g.chunk;
        int nt = std::min(g.chunk, first + count - t0);
        size_t off = (size_t)t0 * tp, bytes = (size_t)nt * tp * sizeof(float);
        cudaEvent_t* e = &g.ev[(size_t)4 * c];
        CK(cudaEventRecord(e[0], g.st[s]));
        CK(cudaMemcpyAsync(g.dA[s], h_in + off, bytes, cudaMemcpyHostToDevice, g.st[s]));
        CK(cudaEventRecord(e[1], g.st[s]));
        float *a = g.dA[s], *b = g.dB[s];
        for (int p = 0; p < P; p++) { launch_blur(a, b, T, nt, g.st[s]); std::swap(a, b); }
        CK(cudaGetLastError());
        CK(cudaEventRecord(e[2], g.st[s]));
        CK(cudaMemcpyAsync(h_out + off, a, bytes, cudaMemcpyDeviceToHost, g.st[s]));   // `a` holds the latest output
        CK(cudaEventRecord(e[3], g.st[s]));
    }
    for (int s = 0; s < g.S; s++) CK(cudaEventRecord(g.evEnd[s], g.st[s]));
    return nch;
}

struct GpuTimes { double h2d = 0, kern = 0, d2h = 0, span = 0; };

static GpuTimes gpu_collect(GpuCtx& g, int nch) {
    GpuTimes r;
    float ms;
    for (int c = 0; c < nch; c++) {
        cudaEvent_t* e = &g.ev[(size_t)4 * c];
        CK(cudaEventElapsedTime(&ms, e[0], e[1])); r.h2d += ms;
        CK(cudaEventElapsedTime(&ms, e[1], e[2])); r.kern += ms;
        CK(cudaEventElapsedTime(&ms, e[2], e[3])); r.d2h += ms;
    }
    for (int s = 0; s < g.S; s++) {
        CK(cudaEventElapsedTime(&ms, g.evStart, g.evEnd[s]));
        r.span = std::max(r.span, (double)ms);
    }
    return r;
}

// ------------------------------------------------------------------ helpers
static void init_input(float* p, size_t n) {
#pragma omp parallel for
    for (long long i = 0; i < (long long)n; i++) {
        uint32_t x = (uint32_t)i * 2654435761u + 12345u;
        x ^= x >> 16; x *= 0x7feb352dU; x ^= x >> 15; x *= 0x846ca68bU; x ^= x >> 16;
        p[i] = (x >> 8) / 16777216.0f;
    }
}

static double max_rel_err(const float* a, const float* ref, size_t n) {
    double mx = 0.0, mr = 1e-6;
#pragma omp parallel for reduction(max : mx, mr)
    for (long long i = 0; i < (long long)n; i++) {
        double d = std::fabs((double)a[i] - (double)ref[i]);
        double r = std::fabs((double)ref[i]);
        if (d > mx) mx = d;
        if (r > mr) mr = r;
    }
    return mx / mr;
}

static std::string get_arg(int argc, char** argv, const char* key, const char* def) {
    for (int i = 1; i + 1 < argc; i++) if (!strcmp(argv[i], key)) return argv[i + 1];
    return def;
}

static std::vector<double> parse_list(const std::string& s) {
    std::vector<double> v;
    size_t p = 0;
    while (p < s.size()) {
        size_t q = s.find(',', p);
        if (q == std::string::npos) q = s.size();
        if (q > p) v.push_back(atof(s.substr(p, q - p).c_str()));
        p = q + 1;
    }
    return v;
}

struct RunResult {
    int nC = 0, nG = 0, team = 0;
    double makespan = 0, sched = 0, cpu_wall = 0, cpu_busy = 0, sync_wait = 0;
    GpuTimes g;
};

// One timed execution of the whole workload with CPU share f_cpu.
static RunResult run_once(GpuCtx& g, const float* h_in, float* h_out, int N, int T, int P, int th, double f_cpu) {
    RunResult r;
    r.nC = (int)llround(f_cpu * N);
    r.nC = std::max(0, std::min(N, r.nC));
    r.nG = N - r.nC;                       // GPU takes tiles [0, nG), CPU takes [nG, N)
    std::vector<double> busy((size_t)(th + 1) * 8, 0.0);
    CK(cudaDeviceSynchronize());           // start from an idle GPU (untimed)

    double t0 = omp_get_wtime();
    int nch = 0;
    if (r.nG > 0) nch = gpu_enqueue(g, h_in, h_out, 0, r.nG, T, P);
    double t_enq = omp_get_wtime();
    if (r.nC > 0) r.team = cpu_process(h_in, h_out, r.nG, r.nC, T, P, th, busy.data());
    double t_cpu = omp_get_wtime();
    if (r.nG > 0) for (int s = 0; s < g.S; s++) CK(cudaStreamSynchronize(g.st[s]));
    double t1 = omp_get_wtime();

    r.makespan = (t1 - t0) * 1e3;
    r.sched = (t_enq - t0) * 1e3;
    r.cpu_wall = r.nC > 0 ? (t_cpu - t_enq) * 1e3 : 0.0;
    r.sync_wait = r.nG > 0 ? (t1 - t_cpu) * 1e3 : 0.0;
    for (int i = 0; i < r.team && i < th; i++) r.cpu_busy += busy[i * 8] * 1e3;
    if (r.nG > 0) r.g = gpu_collect(g, nch);
    return r;
}

int main(int argc, char** argv) {
    int N = atoi(get_arg(argc, argv, "--tiles", "256").c_str());
    int T = atoi(get_arg(argc, argv, "--tile", "256").c_str());
    int P = atoi(get_arg(argc, argv, "--passes", "1").c_str());
    int th = atoi(get_arg(argc, argv, "--threads", "4").c_str());
    int S = atoi(get_arg(argc, argv, "--streams", "1").c_str());
    int chunk = atoi(get_arg(argc, argv, "--chunk", "64").c_str());
    int pinned = atoi(get_arg(argc, argv, "--pinned", "1").c_str());
    int reps = atoi(get_arg(argc, argv, "--reps", "5").c_str());
    int header = atoi(get_arg(argc, argv, "--header", "1").c_str());
    std::string workload = get_arg(argc, argv, "--workload", "tileblur");
    std::vector<double> fracs = parse_list(get_arg(argc, argv, "--fracs", "0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1"));
    if (N < 1 || T < 4 || P < 1 || th < 1 || S < 1 || chunk < 1 || reps < 1 || fracs.empty()) {
        fprintf(stderr, "bad arguments\n");
        return 1;
    }
    omp_set_dynamic(0);
    CK(cudaFree(0));                       // force CUDA context creation before any timing

    size_t tp = (size_t)T * T, n_el = (size_t)N * tp, bytes = n_el * sizeof(float);
    float *h_in = nullptr, *h_out = nullptr;
    if (pinned) { CK(cudaHostAlloc((void**)&h_in, bytes, cudaHostAllocDefault)); CK(cudaHostAlloc((void**)&h_out, bytes, cudaHostAllocDefault)); }
    else { h_in = (float*)malloc(bytes); h_out = (float*)malloc(bytes); }
    std::vector<float> ref(n_el);
    if (!h_in || !h_out) { fprintf(stderr, "host allocation failed\n"); return 1; }
    init_input(h_in, n_el);

    int maxth = omp_get_max_threads();
    std::vector<double> busy0((size_t)(std::max(1, maxth) + 1) * 8, 0.0);
    cpu_process(h_in, ref.data(), 0, N, T, P, std::max(1, maxth), busy0.data());   // CPU reference result

    GpuCtx g;
    gpu_setup(g, S, chunk, N, tp);
    double dev_bytes = (double)S * 2 * chunk * tp * sizeof(float);

    if (header)
        printf("workload,tiles,tile_dim,passes,threads_req,threads_actual,streams,chunk_tiles,pinned,cpu_frac,tiles_cpu,tiles_gpu,rep,"
               "makespan_ms,sched_ms,cpu_wall_ms,cpu_busy_sum_ms,gpu_h2d_ms,gpu_kernel_ms,gpu_d2h_ms,gpu_span_ms,sync_wait_ms,"
               "max_rel_err,host_bytes,device_bytes\n");

    // warm-up: GPU clocks, OpenMP thread pool, page faults
    const double warm[3] = {0.0, 0.5, 1.0};
    for (double w : warm) run_once(g, h_in, h_out, N, T, P, th, w);

    for (int rep = 0; rep < reps; rep++) {
        std::vector<double> order = fracs;
        if (rep & 1) std::reverse(order.begin(), order.end());   // alternate sweep direction to spread drift
        for (double f : order) {
            bool check = (rep == 0);
            if (check) memset(h_out, 0, bytes);
            RunResult r = run_once(g, h_in, h_out, N, T, P, th, f);
            char errbuf[32];
            if (check) snprintf(errbuf, sizeof errbuf, "%.3e", max_rel_err(h_out, ref.data(), n_el));
            else snprintf(errbuf, sizeof errbuf, "nan");
            printf("%s,%d,%d,%d,%d,%d,%d,%d,%d,%.4f,%d,%d,%d,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%.6f,%s,%.0f,%.0f\n",
                   workload.c_str(), N, T, P, th, r.team, S, chunk, pinned, f, r.nC, r.nG, rep + 1,
                   r.makespan, r.sched, r.cpu_wall, r.cpu_busy, r.g.h2d, r.g.kern, r.g.d2h, r.g.span, r.sync_wait,
                   errbuf, 2.0 * bytes, dev_bytes);
            fflush(stdout);
        }
        fprintf(stderr, "[%s N=%d P=%d th=%d] rep %d/%d done\n", workload.c_str(), N, P, th, rep + 1, reps);
    }

    gpu_teardown(g);
    if (pinned) { CK(cudaFreeHost(h_in)); CK(cudaFreeHost(h_out)); } else { free(h_in); free(h_out); }
    return 0;
}

Writing s03_bench.cu


In [6]:
%%writefile run_experiment.py
#!/usr/bin/env python3
"""
S03 (CS7) CPU-GPU workload partitioning: experiment driver.

Compiles s03_bench.cu, records the environment, sweeps
    passes x problem size (tiles) x CPU thread count x CPU share,
repeats every point, and stores one raw row per run in results/s03_raw_runs.csv.
Derived metrics (speedup, efficiency, utilisation) are NOT computed here; analyze.py does that
from the raw rows.

Usage (from the folder containing s03_bench.cu):
    python run_experiment.py --quick          # 2-4 minute sanity run
    python run_experiment.py                  # full experiment (defaults below)
    python run_experiment.py --student-id 23BCE0434 --batch-id B07
"""
import argparse, csv, datetime, io, os, platform, re, shutil, subprocess, sys, time
from pathlib import Path


def sh(cmd, check=False, env=None):
    try:
        cp = subprocess.run(cmd, shell=isinstance(cmd, str), text=True, capture_output=True, env=env)
    except Exception as e:  # tool missing
        return 1, "", str(e)
    if check and cp.returncode != 0:
        print(cp.stdout, cp.stderr, file=sys.stderr)
        raise SystemExit(f"command failed: {cmd}")
    return cp.returncode, cp.stdout.strip(), cp.stderr.strip()


def detect_environment():
    info = {}
    rc, out, _ = sh("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader")
    if rc == 0 and out:
        first = out.splitlines()[0].split(",")
        info["gpu_name"] = first[0].strip()
        info["gpu_mem"] = first[1].strip() if len(first) > 1 else "?"
        info["driver"] = first[2].strip() if len(first) > 2 else "?"
        cc = first[3].strip() if len(first) > 3 else ""
        info["arch"] = "sm_" + cc.replace(".", "") if re.match(r"^\d+\.\d+$", cc) else "sm_75"
        info["gpu_count"] = len(out.splitlines())
    else:
        info.update(gpu_name="none detected", gpu_mem="?", driver="?", arch="sm_75", gpu_count=0)
    cpu = "unknown"
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                cpu = line.split(":", 1)[1].strip()
                break
    except OSError:
        cpu = platform.processor() or "unknown"
    info["cpu_name"] = cpu
    info["logical_cpus"] = os.cpu_count() or 1
    mem_kb = avail_kb = 0
    try:
        for line in open("/proc/meminfo"):
            if line.startswith("MemTotal"):
                mem_kb = int(line.split()[1])
            if line.startswith("MemAvailable"):
                avail_kb = int(line.split()[1])
    except OSError:
        pass
    info["ram_gb"] = round(mem_kb / 1048576, 1)
    info["ram_avail_gb"] = round(avail_kb / 1048576, 1)
    info["nvcc"] = sh("nvcc --version")[1].replace("\n", " | ")
    info["gcc"] = sh("g++ --version")[1].splitlines()[0] if shutil.which("g++") else "g++ not found"
    info["os"] = platform.platform()
    info["python"] = sys.version.split()[0]
    return info


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tiles", type=int, nargs="+", default=[256, 1024, 4096], help="problem sizes: number of tiles")
    ap.add_argument("--tile", type=int, default=256, help="tile edge length in pixels (floats)")
    ap.add_argument("--passes", type=int, nargs="+", default=[1, 16], help="blur passes per tile (arithmetic intensity)")
    ap.add_argument("--threads", type=int, nargs="+", default=[1, 2, 4], help="OpenMP threads for the CPU share")
    ap.add_argument("--streams", type=int, nargs="+", default=[1], help="CUDA streams (1 = serialised offload)")
    ap.add_argument("--pinned", type=int, nargs="+", default=[1], help="1 = pinned host memory, 0 = pageable")
    ap.add_argument("--chunk", type=int, default=64, help="tiles per transfer chunk")
    ap.add_argument("--reps", type=int, default=7)
    ap.add_argument("--fine", action="store_true", help="CPU share step 0.05 instead of 0.1")
    ap.add_argument("--quick", action="store_true", help="small sanity sweep, do not use for reported results")
    ap.add_argument("--skip-compile", action="store_true")
    ap.add_argument("--cpu-only-build", action="store_true", help="build the host-only shim (pipeline test, no GPU)")
    ap.add_argument("--student-id", default="23BCE0434")
    ap.add_argument("--batch-id", default="BATCH_ID")
    ap.add_argument("--outdir", default="results")
    ap.add_argument("--arch", default="auto")
    args = ap.parse_args()

    if args.quick:
        args.tiles, args.passes, args.threads, args.reps = [64, 256], [1, 8], [1, 4], 3

    root = Path(__file__).resolve().parent
    src = root / "s03_bench.cu"
    exe = root / ("s03_bench_cpu" if args.cpu_only_build else "s03_bench")
    out = Path(args.outdir)
    out.mkdir(parents=True, exist_ok=True)
    if not src.exists():
        raise SystemExit("s03_bench.cu must be in the same folder as this script")

    env_info = detect_environment()
    arch = env_info["arch"] if args.arch == "auto" else args.arch
    threads = [t for t in args.threads if t <= env_info["logical_cpus"]]
    if threads != args.threads:
        print(f"note: dropping thread counts above the {env_info['logical_cpus']} logical CPUs available -> {threads}")
    if not threads:
        threads = [env_info["logical_cpus"]]

    if not args.skip_compile:
        if args.cpu_only_build:
            compile_cmd = ["g++", "-O3", "-fopenmp", "-march=native", "-DNO_CUDA", "-x", "c++", str(src), "-o", str(exe)]
        else:
            compile_cmd = ["nvcc", "-O3", "-std=c++14", f"-arch={arch}", "-Xcompiler", "-fopenmp,-O3,-march=native",
                           str(src), "-o", str(exe)]
        print("+", " ".join(compile_cmd), flush=True)
        sh(compile_cmd, check=True)
    else:
        compile_cmd = ["(skipped; reused existing binary)"]
    if not exe.exists():
        raise SystemExit("binary missing; compile first")

    hardware = f"{env_info['gpu_name']} + {env_info['cpu_name']} ({env_info['logical_cpus']} logical CPUs)"
    if args.cpu_only_build:
        hardware = "NO GPU (host-only shim) + " + env_info["cpu_name"]
    (out / "system_info.txt").write_text(
        "S03 environment record\n"
        f"Date: {datetime.datetime.now().isoformat(timespec='seconds')}\n"
        f"GPU: {env_info['gpu_name']} | memory {env_info['gpu_mem']} | driver {env_info['driver']} | target {arch} | GPUs visible: {env_info['gpu_count']}\n"
        f"CPU: {env_info['cpu_name']} | logical CPUs {env_info['logical_cpus']}\n"
        f"RAM: {env_info['ram_gb']} GB total, {env_info['ram_avail_gb']} GB available at start\n"
        f"OS: {env_info['os']}\nPython: {env_info['python']}\n"
        f"CUDA compiler: {env_info['nvcc']}\nHost compiler: {env_info['gcc']}\n"
        f"Compile command: {' '.join(compile_cmd)}\n"
        f"Runtime: OMP_DYNAMIC=FALSE, CUDA_VISIBLE_DEVICES=0, OMP thread counts {threads}\n"
        f"Sweep: tiles={args.tiles} tile={args.tile} passes={args.passes} streams={args.streams} pinned={args.pinned} "
        f"chunk={args.chunk} reps={args.reps} fine={args.fine} quick={args.quick}\n"
    )

    step = 0.05 if args.fine else 0.1
    nfrac = int(round(1 / step))
    fracs = ",".join(f"{i * step:.2f}" for i in range(nfrac + 1))

    run_env = os.environ.copy()
    run_env["OMP_DYNAMIC"] = "FALSE"
    run_env.setdefault("CUDA_VISIBLE_DEVICES", "0")

    configs = [(p, n, th, st, pin) for p in args.passes for n in args.tiles for th in threads
               for st in args.streams for pin in args.pinned]
    raw_path = out / "s03_raw_runs.csv"
    if raw_path.exists():
        raw_path.rename(out / f"s03_raw_runs_previous_{int(time.time())}.csv")
    cols = None
    t_start = time.time()
    run_counter = 0
    print(f"{len(configs)} configurations x {nfrac + 1} CPU shares x {args.reps} repetitions", flush=True)

    for ci, (p, n, th, st, pin) in enumerate(configs, 1):
        need_gb = 3.2 * n * args.tile * args.tile * 4 / 1e9
        if env_info["ram_avail_gb"] and need_gb > env_info["ram_avail_gb"] * 0.8:
            print(f"skip tiles={n}: needs about {need_gb:.1f} GB host RAM, only {env_info['ram_avail_gb']} GB available")
            continue
        cmd = [str(exe), "--tiles", str(n), "--tile", str(args.tile), "--passes", str(p), "--threads", str(th),
               "--streams", str(st), "--chunk", str(args.chunk), "--pinned", str(pin), "--reps", str(args.reps),
               "--fracs", fracs, "--workload", "tileblur"]
        t0 = time.time()
        cp = subprocess.run(cmd, text=True, capture_output=True, env=run_env)
        if cp.returncode != 0:
            print(cp.stdout[-2000:], cp.stderr[-2000:], file=sys.stderr)
            raise SystemExit(f"benchmark failed for {cmd}")
        rows = list(csv.DictReader(io.StringIO(cp.stdout)))
        cfg_id = f"p{p}_n{n}_t{th}_s{st}_pin{pin}"
        for r in rows:
            run_counter += 1
            r["Student_ID"] = args.student_id
            r["Batch_ID"] = args.batch_id
            r["Dataset"] = f"synthetic_tiles_{args.tile}x{args.tile}_float32"
            r["Hardware"] = hardware
            r["Run_ID"] = f"{cfg_id}_f{float(r['cpu_frac']):.2f}_r{r['rep']}"
        if cols is None:
            cols = ["Run_ID", "Student_ID", "Batch_ID", "Dataset", "Hardware"] + [c for c in rows[0].keys()
                                                                                    if c not in ("Run_ID", "Student_ID", "Batch_ID", "Dataset", "Hardware")]
        with open(raw_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols)
            if f.tell() == 0:
                w.writeheader()
            w.writerows(rows)
        bad = [r for r in rows if r["max_rel_err"] != "nan" and float(r["max_rel_err"]) > 1e-5]
        dt = time.time() - t0
        done = time.time() - t_start
        eta = done / ci * (len(configs) - ci)
        flag = f"  WARNING: {len(bad)} runs failed the correctness check" if bad else ""
        print(f"[{ci}/{len(configs)}] {cfg_id}: {len(rows)} rows in {dt:.0f}s, elapsed {done / 60:.1f} min, ETA {eta / 60:.1f} min{flag}", flush=True)

    print(f"\nRaw rows -> {raw_path} ({run_counter} runs)\nEnvironment -> {out / 'system_info.txt'}\nNext: python analyze.py")


if __name__ == "__main__":
    main()

Writing run_experiment.py


In [14]:
!python run_experiment.py --quick --outdir results_quick --student-id {STUDENT_ID} --batch-id {BATCH_ID}
!python analyze.py --input results_quick/s03_raw_runs.csv --outdir results_quick

# import subprocess, time, os
# env = dict(os.environ, OMP_DYNAMIC="FALSE", CUDA_VISIBLE_DEVICES="0")
# fr = "0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1"
# for passes, tiles, th in [(1,64,1),(1,64,4),(8,64,1),(8,64,4),(1,256,1),(1,256,4),(8,256,1),(8,256,4)]:
#     cmd = f"./s03_bench --tiles {tiles} --tile 256 --passes {passes} --threads {th} --reps 1 --fracs {fr} --header 0"
#     s = time.time()
#     try:
#         p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=90, env=env)
#         print(f"passes={passes} tiles={tiles} threads={th}: exit {p.returncode}, {time.time()-s:.1f}s, {len(p.stdout.splitlines())} rows")
#         if p.returncode: print(p.stderr[-500:])
#     except subprocess.TimeoutExpired:
#         print(f"passes={passes} tiles={tiles} threads={th}: TIMED OUT (>90s)")
# print("cpus:", os.cpu_count(), "affinity:", len(os.sched_getaffinity(0)))

+ nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native /kaggle/working/s03_bench.cu -o /kaggle/working/s03_bench
8 configurations x 11 CPU shares x 3 repetitions
[1/8] p1_n64_t1_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[2/8] p1_n64_t4_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[3/8] p1_n256_t1_s1_pin1: 33 rows in 1s, elapsed 0.0 min, ETA 0.1 min
[4/8] p1_n256_t4_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.1 min
[5/8] p8_n64_t1_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.0 min
[6/8] p8_n64_t4_s1_pin1: 33 rows in 1s, elapsed 0.1 min, ETA 0.0 min
[7/8] p8_n256_t1_s1_pin1: 33 rows in 3s, elapsed 0.1 min, ETA 0.0 min
[8/8] p8_n256_t4_s1_pin1: 33 rows in 2s, elapsed 0.1 min, ETA 0.0 min

Raw rows -> results_quick/s03_raw_runs.csv (264 runs)
Environment -> results_quick/system_info.txt
Next: python analyze.py
264 runs, 8 configurations, correctness failures: 0 (max error 0.00e+00)
 passes  tiles  threads_req  gpu_only_ms  cpu_only_ms  best_frac  be

In [15]:
!python run_experiment.py --student-id {STUDENT_ID} --batch-id {BATCH_ID}

+ nvcc -O3 -std=c++14 -arch=sm_75 -Xcompiler -fopenmp,-O3,-march=native /kaggle/working/s03_bench.cu -o /kaggle/working/s03_bench
18 configurations x 11 CPU shares x 7 repetitions
[1/18] p1_n256_t1_s1_pin1: 77 rows in 2s, elapsed 0.0 min, ETA 0.5 min
[2/18] p1_n256_t2_s1_pin1: 77 rows in 1s, elapsed 0.1 min, ETA 0.4 min
[3/18] p1_n256_t4_s1_pin1: 77 rows in 1s, elapsed 0.1 min, ETA 0.4 min
[4/18] p1_n1024_t1_s1_pin1: 77 rows in 5s, elapsed 0.2 min, ETA 0.6 min
[5/18] p1_n1024_t2_s1_pin1: 77 rows in 5s, elapsed 0.2 min, ETA 0.6 min
[6/18] p1_n1024_t4_s1_pin1: 77 rows in 4s, elapsed 0.3 min, ETA 0.6 min
[7/18] p1_n4096_t1_s1_pin1: 77 rows in 20s, elapsed 0.6 min, ETA 1.0 min
[8/18] p1_n4096_t2_s1_pin1: 77 rows in 16s, elapsed 0.9 min, ETA 1.1 min
[9/18] p1_n4096_t4_s1_pin1: 77 rows in 16s, elapsed 1.2 min, ETA 1.2 min
[10/18] p16_n256_t1_s1_pin1: 77 rows in 7s, elapsed 1.3 min, ETA 1.0 min
[11/18] p16_n256_t2_s1_pin1: 77 rows in 4s, elapsed 1.4 min, ETA 0.9 min
[12/18] p16_n256_t4_s1_pin

In [16]:
!python analyze.py

1386 runs, 18 configurations, correctness failures: 0 (max error 0.00e+00)
 passes  tiles  threads_req  gpu_only_ms  cpu_only_ms  best_frac  best_ms  speedup_best_vs_gpu_only  predicted_frac  hybrid_beats_gpu_only
      1    256            1       11.589       15.570        0.4    7.073                     1.638           0.427                   True
      1    256            2       11.875        8.426        0.6    5.249                     2.262           0.585                   True
      1    256            4       11.261        6.514        0.6    4.524                     2.489           0.634                   True
     16    256            1       21.353      156.383        0.1   19.216                     1.111           0.120                  False
     16    256            2       20.020       85.980        0.2   17.792                     1.125           0.189                  False
     16    256            4       20.025       78.301        0.2   16.167                  